Part 1: Getting V1 + higher-order-area spikes during drifting gratings
¶
Assignment step:
Download spike data from 2 sessions from V1 and 1 additional higher-order area (HOA) during the moving (drifting) gratings stimulus.
By the end of this notebook you will have, for each of 2 sessions:
every spike from good V1 and HOA units during drifting gratings, aligned to each grating presentation
a
trials × units firing-rate table
, which Part 2 (tuning curves + correlation heatmaps) runs on
Roadmap
#
Step
Downloads?
0
What is this dataset? (mental model)
no
1
Set up the cache
tiny metadata
2
Sessions table, and why only one session type
no
3
Probes → channels → units: the hierarchy in practice
no
4
Unit quality metrics (which "neurons" to trust)
no
5
Brain areas, and choosing the HOA + 2 sessions
no
6
Pre-checks: grating responsiveness and receptive-field overlap
~small CSV
7
Download the 2 sessions
~2–3 GB each
8
Tour of a session object + the drifting-grating stimulus
no
9
Extract, bin, and save spikes
no
10
Sanity checks: raster and PSTH
no
Environment note:
allensdk
needs Python
3.8–3.11
; it does not install on 3.13. If needed, create one:
conda create -n allen python=3.10
→
conda activate allen
→
pip install allensdk ipykernel
, then select that kernel in VS Code.

0. The mental model: what is this dataset?
¶
Allen Brain Observatory – Visual Coding – Neuropixels.
A mouse is head-fixed on a running wheel in front of a screen. Up to
6 Neuropixels probes
go into its brain at the same time, mostly into visual cortex, with the deep end reaching hippocampus and thalamus. While the mouse watches a ~3 hour "playlist" of visual stimuli, the probes record the electrical activity of hundreds of neurons at once. Across 58 sessions this adds up to roughly 100,000 units.
🧱 Minecraft analogy for the data hierarchy
¶
Think of the mouse brain as a
Minecraft world
:
Dataset level
Minecraft version
ID column
session
one world save: one mouse, one recording day
ecephys_session_id
probe
a 1-block-wide
mineshaft
drilled straight down through the layers (grass → dirt → stone → deepslate ≈ cortex → hippocampus → thalamus)
ecephys_probe_id
channel
a
sensor block
placed every few blocks along the shaft (~384 per probe, ~20 µm apart)
ecephys_channel_id
unit
a
mob
the sensors heard. Each mob is heard by several nearby sensors, loudest at one; spike sorting works out which mob made which sound
unit_id
(the index)
structure / area
the
biome
around the loudest sensor (VISp, VISl, CA1, LGd, …)
ecephys_structure_acronym
So a unit "belongs" to an area because the channel where its spikes were biggest sits in that area.
⚽ Soccer analogy for spikes + stimuli
¶
A session is a
3-hour match
.
The
stimulus table
is the match
event log
:
"t = 1234.5 s: drifting grating starts, direction 90°, temporal frequency 2 Hz, lasts 2 s."
Spike times
are timestamps of every touch of the ball by every player (unit).
Nearly every analysis is
"what did each player do in the 2 s after each kickoff of type X?"
, i.e. aligning spike times to stimulus events. Everything in Parts 1 and 2 is a version of this.

1. Set up the cache
¶
EcephysProjectCache
is the SDK's front door. It works like a game that
streams map chunks on demand
:
Metadata
(sessions/probes/channels/units tables) is the
world map
: small, downloaded once, and already in
ecephys_data/
(
sessions.csv
,
units.csv
, …).
Session NWB files
are the
chunks themselves
: big (2–3 GB) and downloaded only when you "walk into" a session with
get_session_data()
.
manifest.json
is the save index. It records what's already on disk, so nothing is downloaded twice.
⚠️
Disk/OneDrive:
this folder is inside OneDrive, so OneDrive will try to upload the ~5 GB of NWB files. Your C: drive has ~20 GB free. To avoid syncing, point
DATA_DIR
at a folder outside OneDrive, e.g.
Path("C:/allen_data")
. Metadata will re-download there, which takes seconds.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from allensdk.brain_observatory.ecephys.ecephys_project_cache import EcephysProjectCache

DATA_DIR = Path("ecephys_data")     # change to a non-OneDrive folder if you prefer (see note above)
cache = EcephysProjectCache.from_warehouse(manifest=DATA_DIR / "manifest.json")

# consistent colors for the whole project: V1 = blue, HOA = orange
COLORS = {"V1": "#2a78d6", "HOA": "#eb6834"}
MUTED = "#52514e"


2. The sessions table
¶
One row per recording session (= per mouse). Useful columns:
session_type
: which stimulus playlist the mouse saw (see below)
full_genotype
: wild type, or a Cre line (Sst/Pvalb/Vip) used to optogenetically tag inhibitory cell types
unit_count
,
probe_count
,
ecephys_structure_acronyms
: what was recorded

In [ ]:
sessions = cache.get_session_table()
print(len(sessions), "sessions")
print(sessions.session_type.value_counts(), "\n")
sessions[["session_type", "full_genotype", "sex", "age_in_days", "unit_count", "probe_count"]].head()


Two session types = two different "game modes"
¶
brain_observatory_1.1
functional_connectivity
drifting gratings
drifting_gratings
: 8 directions × 5 temporal frequencies × 15 repeats
drifting_gratings_75_repeats
(4 directions × 1 TF × 75 repeats) + a contrast series
design goal
broad survey of many stimuli
many repeats of fewer stimuli
⚽ Comparing players across session types is like comparing stats from two tournaments with
different rules
. Pick one type and stick with it. We use
brain_observatory_1.1
because its 8 directions give a full tuning curve, which Part 2 needs.

In [ ]:
SESSION_TYPE = "brain_observatory_1.1"
DG_STIM = "drifting_gratings"

bo_sessions = sessions[sessions.session_type == SESSION_TYPE]
print(len(bo_sessions), "sessions of type", SESSION_TYPE)


3. Probes → channels → units: the hierarchy in practice
¶
Let's look at
one
session, drilling down level by level, all from the metadata (no download).

In [ ]:
probes = cache.get_probes()
channels = cache.get_channels()

example_sid = bo_sessions.index[0]

# Level 1: the mineshafts (probes) in this session
print("Probes in session", example_sid)
display(probes[probes.ecephys_session_id == example_sid][["name", "channel_count", "unit_count", "ecephys_structure_acronyms"]])

# Level 2: the sensor blocks (channels) along one probe, top of the shaft to the bottom
one_probe = probes[probes.ecephys_session_id == example_sid].index[0]
ch = channels[channels.ecephys_probe_id == one_probe].sort_values("probe_vertical_position", ascending=False)
print(f"\nProbe {one_probe}: which 'biome' each run of channels sits in (top → bottom):")
# collapse consecutive channels in the same structure into runs
runs = (ch.ecephys_structure_acronym.fillna("none") != ch.ecephys_structure_acronym.fillna("none").shift()).cumsum()
print(ch.groupby(runs).agg(structure=("ecephys_structure_acronym", "first"), n_channels=("ecephys_structure_acronym", "size")).to_string(index=False))


Reading the output: going down a probe you typically pass through
visual cortex (VIS…)
, then
hippocampus (CA1, DG, …)
, then
thalamus (LGd, LP, …)
.
NaN
/
none
means the channel is outside the brain or unlabeled. That's the mineshaft passing through the biome layers.

4. Unit quality metrics: which "neurons" can we trust?
¶
What spike sorting is, and how it goes wrong
¶
Each channel picks up spikes from several nearby neurons mixed together.
Spike sorting
separates them by their
waveform shape
, the "voice signature" of each neuron.
🏟️
Analogy:
a few microphones in a stadium, and software that separates individual fans' voices. Three ways it can fail, each with its own metric:
Failure
Metric
Rule (Allen defaults)
Analogy
Two neurons merged into one "unit" (
contamination
)
isi_violations
< 0.5
Neurons have a
refractory period
(~1.5 ms) after firing, like an
ability cooldown
in a game. A real neuron
cannot
fire twice inside its cooldown. If a "unit" does, some of those spikes must come from someone else. The metric estimates the contamination rate relative to the unit's own spikes, so it can exceed 1.
Quiet spikes not detected (
missed spikes
)
amplitude_cutoff
< 0.1
Spike sorting only "hears" spikes louder than a threshold. If the unit's amplitude histogram looks
cut off on the left
, part of it is below the threshold. The value ≈ fraction of spikes missed (max 0.5). Like a minimap that only shows mobs above some volume.
Unit drifts in/out of range (
instability
)
presence_ratio
> 0.95
The recording is split into 100 chunks; this is the fraction of chunks where the unit fired. ⚽ A player on the pitch the
whole match
, not subbed off halfway because the probe drifted.
Important:
cache.get_units()
already applies
these three filters by default. Below we load the
unfiltered
table to see what gets thrown away.
⚠️ From the quality-metrics tutorial: these thresholds are a reasonable starting point, but they're
biased
. Low-firing or small neurons fail more often, e.g.
presence_ratio
for sparse firers. Quote them on your slides as "we used Allen's default QC".

In [ ]:
all_units = cache.get_units(amplitude_cutoff_maximum=np.inf,
                            presence_ratio_minimum=-np.inf,
                            isi_violations_maximum=np.inf)
all_units = all_units[all_units.ecephys_session_id.isin(bo_sessions.index)]

QC = dict(isi_violations_maximum=0.5, amplitude_cutoff_maximum=0.1, presence_ratio_minimum=0.95)
passes = ((all_units.isi_violations < QC["isi_violations_maximum"]) &
          (all_units.amplitude_cutoff < QC["amplitude_cutoff_maximum"]) &
          (all_units.presence_ratio > QC["presence_ratio_minimum"]))
print(f"{len(all_units)} units in {SESSION_TYPE} sessions; {passes.mean():.0%} pass all three QC rules")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
specs = [("isi_violations", 0.5, (0, 2), "keep left of line"),
         ("amplitude_cutoff", 0.1, (0, 0.5), "keep left of line"),
         ("presence_ratio", 0.95, (0.5, 1), "keep right of line")]
for ax, (col, thr, rng, note) in zip(axes, specs):
    vals = all_units[col].clip(*rng)
    ax.hist(vals, bins=60, range=rng, color=COLORS["V1"], edgecolor="white", linewidth=0.5)
    ax.axvline(thr, color=MUTED, ls="--", lw=1.5)
    ax.set(title=col, xlabel=f"value (threshold {thr}; {note})", ylabel="units")
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()


Reading the histograms:
most units cluster on the "good" side of each line. The long tails are what QC removes. Pile-ups at the edges of
isi_violations
/
amplitude_cutoff
are values clipped into the plot range.

5. Brain areas, and choosing the HOA
¶
Visual areas in the mouse
¶
Acronym
Name
Rough hierarchy level
VISp
V1
(primary visual cortex)
1 (first cortical stop)
VISl
LM (lateromedial)
2
VISrl
RL (rostrolateral)
2
VISal
AL (anterolateral)
3
VISpm
PM (posteromedial)
4
VISam
AM (anteromedial)
5 (highest)
⚽
The visual hierarchy as a passing build-up:
the eye → thalamus (LGd) is the
goalkeeper
that starts play.
V1
is the
defense
, first to receive the ball, with precise local reads like "there's an edge at 45° right here". It passes to
midfield (LM, RL, AL)
, then to
attack (PM, AM)
. Each pass:
adds
delay
, so higher areas respond later
combines information from more of the field: bigger receptive fields, broader and more abstract tuning
Siegle et al. 2021 (
Nature
, the paper behind this dataset) measured this ordering from the data itself. It's the main reason "V1 vs HOA" is an interesting comparison.
Choosing the HOA:
LM (
VISl
)
is the closest to V1: similar but somewhat different, and the safest choice.
AM (
VISam
)
sits at the top: the biggest contrast with V1.
Below, count good units per area per session to see which HOA has enough units in enough sessions.

In [ ]:
AREAS = {"VISp": "V1", "VISl": "LM", "VISrl": "RL", "VISal": "AL", "VISpm": "PM", "VISam": "AM"}

units = cache.get_units(**QC)   # QC'd units (same as the default filter, but explicit)
bo_units = units[units.ecephys_session_id.isin(bo_sessions.index) &
                 units.ecephys_structure_acronym.isin(AREAS)]

unit_counts = (bo_units.groupby(["ecephys_session_id", "ecephys_structure_acronym"]).size()
                       .unstack(fill_value=0)
                       .reindex(columns=list(AREAS), fill_value=0)
                       .rename(columns=AREAS))
unit_counts.sort_values("V1", ascending=False).head(10)


In [ ]:
MIN_UNITS = 30   # need enough units per area for a meaningful correlation matrix

summary = {}
for hoa in [a for a in AREAS.values() if a != "V1"]:
    ok = (unit_counts["V1"] >= MIN_UNITS) & (unit_counts[hoa] >= MIN_UNITS)
    summary[hoa] = {"n_sessions_ok": int(ok.sum()),
                    "median_HOA_units": unit_counts.loc[ok, hoa].median()}
pd.DataFrame(summary).T.sort_values("n_sessions_ok", ascending=False)


In [ ]:
HOA = "VISl"                 # <-- your choice (e.g. "VISl" = LM, "VISam" = AM)
HOA_NAME = AREAS[HOA]

cand = unit_counts[(unit_counts["V1"] >= MIN_UNITS) & (unit_counts[HOA_NAME] >= MIN_UNITS)][["V1", HOA_NAME]].copy()
cand["min_units"] = cand.min(axis=1)    # rank by the weaker area so one big area can't carry a session
cand = cand.join(bo_sessions[["full_genotype", "sex", "age_in_days"]]).sort_values("min_units", ascending=False)
cand.head(8)


6. Pre-checks before spending GBs: responsiveness and receptive fields
¶
Allen precomputed per-unit response metrics for every stimulus. These come as one CSV, so we can
scout sessions without downloading them
.
Column
Meaning
firing_rate_dg
mean firing rate during drifting gratings (Hz)
g_osi_dg
global orientation selectivity index
(0 = responds equally to all orientations, 1 = only one)
pref_ori_dg
,
pref_tf_dg
preferred direction and temporal frequency
azimuth_rf
,
elevation_rf
receptive-field center on the screen (degrees)
p_value_rf
is the RF real? small = a significant RF
🧭
gOSI analogy:
each orientation's response is an
arrow
pointing at that angle, with length = firing rate. Add all the arrows. If the unit responds equally to every angle, they cancel and gOSI ≈ 0. If one angle dominates, the sum points there and gOSI → 1.
Receptive fields (from the RF tutorial)
¶
📹 Each neuron is a
security camera
that only watches one patch of the screen: its
receptive field (RF)
, mapped with small Gabor patches flashed at a 9×9 grid of locations. V1 cameras are zoomed in (small RFs); higher-area cameras are wide-angle (bigger RFs).
Why it matters here:
drifting gratings fill the whole screen, so every unit sees
some
grating. But correlations between V1 and HOA units are easier to interpret if both populations are
watching the same part of the screen
. If V1 watches the top-left and LM watches the bottom-right, low inter-area correlation may just mean "different patch", not "different computation".

In [ ]:
metrics = cache.get_unit_analysis_metrics_by_session_type(SESSION_TYPE)

m = bo_units[["ecephys_session_id", "ecephys_structure_acronym"]].join(
        metrics[["firing_rate_dg", "g_osi_dg", "azimuth_rf", "elevation_rf", "p_value_rf"]], how="inner")
m = m[m.ecephys_session_id.isin(cand.index) & m.ecephys_structure_acronym.isin(["VISp", HOA])]
m["area"] = m.ecephys_structure_acronym.map(AREAS)

dg_summary = m.groupby(["ecephys_session_id", "area"])[["firing_rate_dg", "g_osi_dg"]].median().unstack()
dg_summary.columns = [f"{stat}_{area}" for stat, area in dg_summary.columns]
cand.join(dg_summary).head(8).round(2)


Decide here.
Look for sessions with:
high
min_units
reasonable
firing_rate_dg
(a few Hz or more) in both areas
some orientation selectivity (
g_osi_dg
typically ~0.1–0.4)
Genotype: Cre lines (e.g.
Sst-IRES-Cre…
) are fine for this analysis, but note which mice they were on your slides.
By default we take the top 2. Override by hand if the table suggests a better pair.

In [ ]:
SESSION_IDS = cand.index[:2].tolist()   # or set by hand, e.g. [798911424, 791319847]
print("Chosen sessions:", SESSION_IDS, "| areas: V1 +", HOA_NAME)

# RF overlap check: where on the screen are each area's units looking?
fig, axes = plt.subplots(1, len(SESSION_IDS), figsize=(5.5 * len(SESSION_IDS), 4), sharex=True, sharey=True)
for ax, sid in zip(np.atleast_1d(axes), SESSION_IDS):
    for area in ["V1", HOA_NAME]:
        sub = m[(m.ecephys_session_id == sid) & (m.area == area) & (m.p_value_rf < 0.01)]
        color = COLORS["V1"] if area == "V1" else COLORS["HOA"]
        ax.scatter(sub.azimuth_rf, sub.elevation_rf, s=36, color=color, alpha=0.7,
                   edgecolor="white", linewidth=1, label=f"{area} ({len(sub)} units w/ RF)")
    ax.set(title=f"session {sid}", xlabel="azimuth (deg, left ↔ right)")
    ax.legend(frameon=False, fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)
np.atleast_1d(axes)[0].set_ylabel("elevation (deg, down ↔ up)")
plt.suptitle("Receptive-field centers: do V1 and the HOA watch the same part of the screen?")
plt.tight_layout()


Reading the RF plot:
the points are jittered on the 9×9 grid of RF-mapping positions. Heavy overlap between blue and orange clouds = good. If they're far apart, consider another session, or mention it as a caveat when interpreting V1–HOA correlations.

7. Download the 2 sessions (slow: ~2–3 GB each)
¶
get_session_data()
downloads the NWB file (if not cached) and returns an
EcephysSession
object. We pass the QC thresholds explicitly so the session's
units
table matches what we used above.
If the download times out,
just re-run the cell
. Finished files are skipped thanks to the manifest.

In [ ]:
session_data = {}
for sid in SESSION_IDS:
    print("loading session", sid, "...")
    session_data[sid] = cache.get_session_data(sid, **QC)
print("done")


8. Tour of a session object + the drifting-grating stimulus
¶
An
EcephysSession
holds everything from one recording. The parts we need:
Attribute / method
What it is
s.units
table of QC'd units with area, depth, etc.
s.spike_times
dict
{unit_id: array of spike times (s)}
: the raw "touch log"
s.stimulus_names
/
s.get_stimulus_epochs()
the playlist, and when each block happened
s.get_stimulus_table([...])
one row per presentation of the given stimuli (the event log)
s.presentationwise_spike_times(...)
spikes aligned to each presentation
s.presentationwise_spike_counts(...)
spikes binned in time around each presentation
s.running_speed
wheel speed (running changes firing rates in mouse V1!)

In [ ]:
s = session_data[SESSION_IDS[0]]
print("units per area in this session:")
print(s.units.ecephys_structure_acronym.value_counts().head(12), "\n")

epochs = s.get_stimulus_epochs()
epochs[["stimulus_name", "start_time", "duration"]].head(15)


The epochs table shows the
playlist
. Drifting gratings appear in
several separate blocks
spread across the session, not all at once. ⚽ It's like teams
switching ends at halftime
: spreading conditions across the session keeps slow changes (fatigue, arousal, probe drift) from lining up with one particular stimulus. Within each block the conditions are randomly interleaved for the same reason.
What a drifting grating is
¶
Picture black-and-white
stripes on a treadmill belt
sliding across the screen (sinusoidal, so the edges are soft).
Orientation
= the angle of the stripes.
Direction
= which way they move. They always move perpendicular to the stripes.
⚠️ The column named
orientation
actually holds
direction
: 0°, 45°, …, 315°. 0° and 180° have the
same
stripe angle but move in opposite directions. Neurons that care only about the angle are
orientation-selective
; neurons that also care about the motion are
direction-selective
.
Temporal frequency (TF)
, 1/2/4/8/15 Hz = how many stripes pass a fixed point per second (the belt's speed / tempo).
Fixed: spatial frequency 0.04 cycles/deg (stripe width), contrast 80%, 2 s per presentation, ~1 s of gray screen in between.
null
rows
=
blank trials
(gray screen, no grating). They're the control, giving each unit's
baseline
firing rate.

In [ ]:
dg = s.get_stimulus_table([DG_STIM])
print(len(dg), "drifting-grating presentations")
display(dg[["start_time", "stop_time", "duration", "orientation", "temporal_frequency", "contrast", "stimulus_block"]].head())

# How many repeats per condition? Expect 15 per (direction, TF), plus a set of blanks.
print("\nRepeats per condition (rows = direction, cols = TF):")
pd.crosstab(dg.orientation.astype(str), dg.temporal_frequency.astype(str))


9. Extract, bin, and save spikes
¶
One session only.
From here on we process a single session,
SID
(the top-ranked one, 798911424). This keeps memory use down: processing both ~2.8 GB sessions in one go is where the notebook was freezing. To do the other session later, restart the kernel, set
SID
to it, and re-run this section.
Two complementary views of the same spikes:
(a) Spike times aligned to each presentation:
presentationwise_spike_times
. One row per spike: which unit, which presentation, and
time since that presentation started
. ⚽ For every kickoff, the list of each player's touches with a stopwatch reset to 0 at kickoff. Exact, but awkward for math.
(b) Binned spike counts:
presentationwise_spike_counts
. Chop the time around each presentation into small
bins
(10 ms here) and count spikes per bin.
🎮
Analogy:
replaying the match at
100 fps
(10 ms per frame) and counting touches per frame. Bigger bins = lower frame rate = smoother but blurrier in time.
The result is a
3D array
,
presentations × time bins × units
. 🧱 Like a stack of spreadsheets: each sheet is one trial, with rows = time bins and columns = units.
We use a window from
−0.5 s to +2.5 s
around grating onset:
0.5 s before → baseline (gray screen)
0–2 s → grating on
2–2.5 s → offset response
(c) The table Part 2 needs:
a
trials × units firing-rate table
. For each presentation and unit, count the spikes from 0–2 s and divide by 2 s, giving the rate in Hz. Each row is one trial and each column is one neuron.
Tuning curves come from grouping rows by direction.
Correlation matrices are
np.corrcoef
over the columns.

In [ ]:
OUT = Path("dg_spikes")
OUT.mkdir(exist_ok=True)
BIN = 0.01
BIN_EDGES = np.arange(-0.5, 2.5 + BIN / 2, BIN)
RESP_WIN = (0.0, 2.0)                       # grating-on window used for firing rates

SID = SESSION_IDS[0]                        # the one session processed from here on (798911424, same as Part 2)
# reuse the session if section 7 already loaded it; otherwise open it from the local cache (no re-download)
loaded = globals().get("session_data", {})
s = loaded[SID] if SID in loaded else cache.get_session_data(SID, **QC)

sel = s.units[s.units.ecephys_structure_acronym.isin(["VISp", HOA])].copy()
sel["area"] = sel.ecephys_structure_acronym.map(AREAS)
# order units: all V1 first, then HOA, each by depth. Part 2's heatmap blocks rely on this order.
sel["is_hoa"] = sel.area != "V1"
sel = sel.sort_values(["is_hoa", "probe_vertical_position"]).drop(columns="is_hoa")
dg = s.get_stimulus_table([DG_STIM])

# (a) exact spike times relative to each presentation
spikes = s.presentationwise_spike_times(stimulus_presentation_ids=dg.index.values,
                                        unit_ids=sel.index.values)
spikes = spikes.join(dg[["orientation", "temporal_frequency"]], on="stimulus_presentation_id")
spikes["area"] = spikes.unit_id.map(sel.area)
spikes.to_pickle(OUT / f"session_{SID}_spike_times.pkl")

# (b) binned counts: presentation x time bin x unit (an xarray DataArray)
counts = s.presentationwise_spike_counts(bin_edges=BIN_EDGES,
                                         stimulus_presentation_ids=dg.index.values,
                                         unit_ids=sel.index.values)
t = counts["time_relative_to_stimulus_onset"].values
np.savez_compressed(OUT / f"session_{SID}_spike_counts.npz",
                    counts=counts.values.astype(np.uint8),
                    presentation_ids=counts["stimulus_presentation_id"].values,
                    time=t,
                    unit_ids=counts["unit_id"].values)

# (c) trials x units firing rates (Hz) in the grating-on window
in_win = (t >= RESP_WIN[0]) & (t < RESP_WIN[1])
rates = pd.DataFrame(counts.values[:, in_win, :].sum(axis=1) / (RESP_WIN[1] - RESP_WIN[0]),
                     index=pd.Index(counts["stimulus_presentation_id"].values, name="stimulus_presentation_id"),
                     columns=pd.Index(counts["unit_id"].values, name="unit_id"))
rates.to_pickle(OUT / f"session_{SID}_rates.pkl")

sel.to_csv(OUT / f"session_{SID}_units.csv")
dg.to_csv(OUT / f"session_{SID}_stimulus.csv")

print(f"session {SID}: {sel.area.value_counts().to_dict()} units | {len(dg)} presentations | "
      f"{len(spikes):,} spikes | rates table {rates.shape}")


What got saved in
dg_spikes/
for this session:
File
Contents
Used for
*_rates.pkl
trials × units firing rate (Hz), 0–2 s
Part 2: tuning curves + correlations
*_stimulus.csv
direction / TF / time of each presentation
grouping trials by condition
*_units.csv
unit metadata (area, depth, …), ordered V1 then HOA
labeling and ordering
*_spike_counts.npz
10 ms binned counts, −0.5 to 2.5 s
PSTHs, latency
*_spike_times.pkl
every individual spike
rasters

10. Sanity checks
¶
(a) Raster plot for one V1 unit
¶
A
raster
draws one row per trial and one tick per spike. Sorting trials by direction lets you
see
tuning before computing anything. ⚽ It's like stacking the touch timelines of every kickoff, grouped by the type of play.
We pick the most orientation-selective, reasonably active V1 unit in this session (
SID
).

In [ ]:
sid = SID
spikes = pd.read_pickle(OUT / f"session_{sid}_spike_times.pkl")
stim = pd.read_csv(OUT / f"session_{sid}_stimulus.csv", index_col=0)
units_df = pd.read_csv(OUT / f"session_{sid}_units.csv", index_col=0)

v1_ids = units_df.index[units_df.area == "V1"]
cands = metrics.loc[metrics.index.intersection(v1_ids)]
unit = cands[cands.firing_rate_dg > 2].g_osi_dg.idxmax()

# order trials: by direction (blanks last), then TF
stim["direction"] = pd.to_numeric(stim.orientation, errors="coerce")
order = stim.sort_values(["direction", "temporal_frequency"], na_position="last").index
row_of = pd.Series(np.arange(len(order)), index=order)

u = spikes[spikes.unit_id == unit]
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(u.time_since_stimulus_presentation_onset, row_of[u.stimulus_presentation_id].values,
           s=1.5, color=COLORS["V1"], marker="|")
# label direction groups on the y axis
grp = stim.loc[order, "direction"].fillna(-1).values
starts = np.r_[0, np.flatnonzero(np.diff(grp)) + 1]
for st in starts[1:]:
    ax.axhline(st, color="#d0cfcb", lw=0.8)
ax.set_yticks(starts + np.diff(np.r_[starts, len(grp)]) / 2,
              [("blank" if g < 0 else f"{g:.0f}°") for g in grp[starts]])
ax.invert_yaxis()
ax.set(xlabel="time from grating onset (s)", ylabel="trials grouped by direction",
       title=f"V1 unit {unit}, gOSI = {cands.loc[unit, 'g_osi_dg']:.2f}")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()


What to look for:
one or two direction bands should be visibly
denser
than the rest. For an orientation-selective cell, look for two dense bands 180° apart. The blank band should look like background. That's the tuning curve, before any math.
(b) Population PSTH per area
¶
A
PSTH
(peri-stimulus time histogram) averages the binned counts over trials and units, giving mean firing rate vs. time. ⚽ Overlay every kickoff replay and ask, frame by frame, "how busy was the team on average?"

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
d = np.load(OUT / f"session_{SID}_spike_counts.npz")
units_df = pd.read_csv(OUT / f"session_{SID}_units.csv", index_col=0)
stim = pd.read_csv(OUT / f"session_{SID}_stimulus.csv", index_col=0)

non_blank = np.isin(d["presentation_ids"], stim.index[pd.to_numeric(stim.orientation, errors="coerce").notna()])
for area, color in [("V1", COLORS["V1"]), (HOA_NAME, COLORS["HOA"])]:
    cols = np.isin(d["unit_ids"], units_df.index[units_df.area == area])
    rate = d["counts"][non_blank][:, :, cols].mean(axis=(0, 2)) / BIN
    ax.plot(d["time"], rate, color=color, lw=2, label=f"{area} ({cols.sum()} units)")
ax.axvspan(0, 2, color="#f0efec", zorder=0)
ax.set(title=f"session {SID}", xlabel="time from grating onset (s); shaded = grating on",
       ylabel="mean firing rate (Hz)")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()


What to expect:
Both areas jump up shortly after onset.
V1 should rise a few ms earlier
than the HOA (the "passing delay" in the hierarchy). Zoom in to the 0–0.15 s window to see it.
Some units oscillate at the grating's temporal frequency. Averaging across TFs mostly washes this out.
An "off" bump after 2 s is normal.
If one curve is flat, revisit the session choice in step 6.
✅ Part 1 done: hand-off to Part 2
¶
Reload everything without touching the NWB files:
rates
=
pd
.
read_pickle
(
f
"dg_spikes/session_
{
sid
}
_rates.pkl"
)
# trials x units (Hz)
stim
=
pd
.
read_csv
(
f
"dg_spikes/session_
{
sid
}
_stimulus.csv"
,
index_col
=
0
)
units
=
pd
.
read_csv
(
f
"dg_spikes/session_
{
sid
}
_units.csv"
,
index_col
=
0
)
# V1 first, then HOA
Tuning curves:
join
rates
with
stim.orientation
, drop blanks,
groupby("orientation")
→ mean and 1.96 × SEM.
Correlation heatmap:
C = np.corrcoef(rates[units.index].values.T)
→
plt.imshow(C, cmap="RdBu_r", vmin=-1, vmax=1)
. The V1|HOA block boundary is at
(units.area == "V1").sum()
.

Part 2: Tuning curves + correlation heatmaps (one specimen)
¶
Assignment step:
For each unit, plot average firing rate (with 95% confidence intervals) at each grating orientation, and heat maps for correlations between units.
This part only reads the files saved in
dg_spikes/
during Part 1. It doesn't need
allensdk
or the NWB files, so it runs in seconds. We analyze
one specimen: session
798911424
(V1 + LM).
#
Step
11
Load the saved trials × units rate table
12
Tuning curves with 95% CIs (example units, then
every
unit)
13
Population view: tuning heatmap + selectivity (OSI/DSI) V1 vs LM
14
Correlation heatmaps (N × N): total, signal, noise
15
Interpretation notes for the slides

11. Load one specimen
¶
Recall the core table from Part 1:
rates
, one row per grating presentation (trial) and one column per unit. Each value is that unit's firing rate (Hz) during the 2 s the grating was on.
🧱 Think of it as a big spreadsheet. Everything in Part 2 is just a different way of
slicing that spreadsheet
:
Tuning curve:
group the
rows
by grating direction, then average each column.
Correlation matrix:
compare
columns
with each other. Do two units go up and down together across rows?

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

OUT = Path("dg_spikes")
SID = 798911424                          # the one specimen analyzed in Part 2
FIG_DIR = OUT / "figs"
FIG_DIR.mkdir(exist_ok=True)

rates = pd.read_pickle(OUT / f"session_{SID}_rates.pkl")               # trials x units (Hz)
units_df = pd.read_csv(OUT / f"session_{SID}_units.csv", index_col=0)  # V1 first, then HOA
stim = pd.read_csv(OUT / f"session_{SID}_stimulus.csv", index_col=0).loc[rates.index]

stim["direction"] = pd.to_numeric(stim.orientation, errors="coerce")  # NaN = blank (gray screen) trial
stim["tf"] = pd.to_numeric(stim.temporal_frequency, errors="coerce")
is_blank = stim.direction.isna()

HOA_NAME = [a for a in units_df.area.unique() if a != "V1"][0]
AREA_ORDER = ["V1", HOA_NAME]
COLORS = {"V1": "#2a78d6", HOA_NAME: "#eb6834"}
MUTED = "#52514e"

assert (rates.columns == units_df.index).all()
print(f"session {SID}: rates table {rates.shape} (trials x units)")
print(units_df.area.value_counts().to_dict(), f"| {(~is_blank).sum()} grating trials, {is_blank.sum()} blank trials")


12. Tuning curves with 95% confidence intervals
¶
What we compute, per unit
¶
For each of the 8 directions there are
75 trials
(5 temporal frequencies × 15 repeats). We pool across temporal frequency.
Mean rate
at that direction = the height of the curve.
95% CI
= mean ± t × SEM, where SEM = std / √n and t ≈ 1.99 for n = 75. This is basically the familiar 1.96, with a tiny small-sample correction.
Baseline
(dashed line) = mean rate on the 30 blank gray-screen trials. Points above it mean the grating
drives
the unit; points below mean it
suppresses
it.
⚽
CI analogy:
a striker scores in 30 of 75 matches. Our best guess is 40%, but with only 75 matches the true rate could plausibly be 29–51%. That range is the 95% CI. More matches (trials) or more consistent play (lower variance) → a narrower interval.
Non-overlapping CIs between two directions = strong evidence the unit really prefers one.
⚠️ The
orientation
column is really
direction
(0–315°). 0° and 180° are the same stripes moving opposite ways. A pure
orientation
cell has
two peaks 180° apart
; a
direction
cell has
one
.
Selectivity indices (one number per unit)
¶
OSI (global orientation selectivity):
treat each direction's mean rate as an arrow at angle
2θ
and add them up. Doubling the angle makes 0° and 180° point the same way. OSI = length of the sum / total length:
0 = no preference, 1 = responds to one orientation only.
DSI (global direction selectivity):
same, but with angle
θ
, so opposite directions cancel. High DSI = cares about which way the stripes move.
Is it tuned at all?
A one-way ANOVA across the 8 directions; p < 0.01 = the rate really depends on direction.

In [ ]:
grat = rates[~is_blank]
dirs_per_trial = stim.loc[grat.index, "direction"]
DIRS = np.sort(dirs_per_trial.unique())                # 0, 45, ..., 315

g = grat.groupby(dirs_per_trial)
tc_mean = g.mean()                                     # directions x units
tc_n = g.size()
t_crit = stats.t.ppf(0.975, tc_n - 1)                  # ~1.99 for n = 75
tc_ci = g.sem().mul(t_crit, axis=0)                    # half-width of the 95% CI
baseline = rates[is_blank].mean()

# selectivity indices from the mean tuning curve
theta = np.deg2rad(DIRS)[:, None]
R = tc_mean.values
osi = np.abs((R * np.exp(2j * theta)).sum(0)) / R.sum(0)
dsi = np.abs((R * np.exp(1j * theta)).sum(0)) / R.sum(0)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    p_anova = [stats.f_oneway(*[grat.loc[dirs_per_trial == d, u] for d in DIRS]).pvalue for u in rates.columns]

tuning = pd.DataFrame({"area": units_df.area,
                       "pref_dir": DIRS[R.argmax(0)],
                       "peak_rate": R.max(0),
                       "baseline": baseline,
                       "osi": osi, "dsi": dsi,
                       "p_anova": p_anova}, index=rates.columns)
tuning["tuned"] = tuning.p_anova < 0.01
tuning.round(3).head()


Example units (for the slides)
¶
The 4 most orientation-selective, clearly tuned units in each area (ANOVA p < 0.01, peak ≥ 2 Hz). Error bars are 95% CIs.

In [ ]:
def plot_tuning(unit, ax, color, title=True):
    # mean rate at each direction with 95% CI error bars, blank-trial baseline dashed
    ax.errorbar(DIRS, tc_mean[unit], yerr=tc_ci[unit], color=color, lw=2,
                marker="o", ms=5, capsize=3, elinewidth=1.2)
    ax.axhline(baseline[unit], color=MUTED, ls="--", lw=1)
    ax.set_xticks(DIRS[::2])
    ax.set_xlim(-20, 335)
    ax.set_ylim(bottom=0)
    ax.spines[["top", "right"]].set_visible(False)
    if title:
        ax.set_title(f"unit {unit}\nOSI {tuning.osi[unit]:.2f} · DSI {tuning.dsi[unit]:.2f}", fontsize=9)

N_EX = 4
fig, axes = plt.subplots(2, N_EX, figsize=(3.2 * N_EX, 6), constrained_layout=True)
for row, area in zip(axes, AREA_ORDER):
    pool = tuning[(tuning.area == area) & tuning.tuned & (tuning.peak_rate >= 2)]
    for ax, unit in zip(row, pool.osi.nlargest(N_EX).index):
        plot_tuning(unit, ax, COLORS[area])
    row[0].set_ylabel(f"{area}\nfiring rate (Hz)")
for ax in axes[-1]:
    ax.set_xlabel("grating direction (deg)")
fig.suptitle(f"Tuning Curves, session {SID} (mean ± 95% CI; dashed = blank-screen baseline)")
fig.savefig(FIG_DIR / f"session_{SID}_tuning_examples.png", dpi=200)


Every unit
¶
The assignment asks for a tuning curve
for each unit
, so here is one grid per area. These are too many for a slide, so they're saved as PNGs in
dg_spikes/figs/
for a backup or appendix slide. Units are ordered by preferred direction, so similar cells sit next to each other. Each panel has its own y-axis, because firing rates differ a lot between units.

In [ ]:
def tuning_grid(area, ncols=10):
    ids = tuning[tuning.area == area].sort_values("pref_dir").index
    nrows = int(np.ceil(len(ids) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(1.9 * ncols, 1.6 * nrows),
                             sharex=True, constrained_layout=True)
    for ax in axes.flat[len(ids):]:
        ax.axis("off")
    for ax, unit in zip(axes.flat, ids):
        plot_tuning(unit, ax, COLORS[area], title=False)
        ax.set_title(f"{unit}{'' if tuning.tuned[unit] else ' (n.s.)'}", fontsize=7)
        ax.tick_params(labelsize=6)
    fig.suptitle(f"{area}: all {len(ids)} units, session {SID} (mean ± 95% CI vs direction; "
                 f"dashed = baseline; n.s. = not significantly tuned)")
    fig.savefig(FIG_DIR / f"session_{SID}_tuning_all_{area}.png", dpi=150)
    return fig

for area in AREA_ORDER:
    tuning_grid(area)


13. Population view: how do V1 and the HOA differ in tuning?
¶
(a) Population tuning heatmap
¶
Each row is one unit's tuning curve, scaled so its peak = 1, with rows sorted by preferred direction. This compresses ~90 tuning curves into a single image.
Sharp diagonal stripe
= units are sharply tuned and their preferences tile all directions.
Blurry / washed-out
= broad tuning.
A
second, fainter stripe 180° away
= orientation (not direction) selectivity.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5), constrained_layout=True)
for ax, area, cmap in zip(axes, AREA_ORDER, ["Blues", "Oranges"]):
    ids = tuning[tuning.area == area].sort_values("pref_dir").index
    norm_tc = (tc_mean[ids] / tc_mean[ids].max()).T              # units x directions, peak = 1
    im = ax.imshow(norm_tc, aspect="auto", cmap=cmap, vmin=0, vmax=1, interpolation="nearest")
    ax.set_xticks(range(len(DIRS)), [f"{d:.0f}" for d in DIRS])
    ax.set(title=f"{area} ({len(ids)} units)", xlabel="grating direction (deg)",
           ylabel="units (sorted by preferred direction)")
    fig.colorbar(im, ax=ax, label="rate / unit's peak rate", shrink=0.8)
fig.suptitle(f"Population tuning, session {SID}")
fig.savefig(FIG_DIR / f"session_{SID}_tuning_population.png", dpi=200)


(b) Selectivity: V1 vs HOA
¶
Compare OSI and DSI distributions with
cumulative distribution (ECDF)
plots. For each value x, the curve shows the fraction of units with selectivity ≤ x.
A curve shifted to the right = a more selective area.
The Mann-Whitney U test asks whether one area's values tend to be larger, without assuming normal distributions.

In [ ]:
summary = tuning.groupby("area").agg(n_units=("osi", "size"),
                                     pct_tuned=("tuned", "mean"),
                                     median_peak_hz=("peak_rate", "median"),
                                     median_osi=("osi", "median"),
                                     median_dsi=("dsi", "median")).loc[AREA_ORDER]
summary["pct_tuned"] = (100 * summary.pct_tuned).round(0)
display(summary.round(3))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), constrained_layout=True)
for ax, metric, label in zip(axes, ["osi", "dsi"], ["orientation selectivity (OSI)", "direction selectivity (DSI)"]):
    a, b = (tuning.loc[tuning.area == ar, metric].dropna() for ar in AREA_ORDER)
    p = stats.mannwhitneyu(a, b).pvalue
    for area, vals in zip(AREA_ORDER, (a, b)):
        x = np.sort(vals)
        ax.step(x, np.arange(1, len(x) + 1) / len(x), where="post", color=COLORS[area], lw=2,
                label=f"{area} (median {np.median(x):.2f})")
    ax.set(xlabel=label, ylabel="fraction of units ≤ x", title=f"Mann-Whitney p = {p:.3g}", ylim=(0, 1.02))
    ax.legend(frameon=False, loc="lower right")
    ax.spines[["top", "right"]].set_visible(False)
fig.savefig(FIG_DIR / f"session_{SID}_selectivity.png", dpi=200)


14. Correlation heatmaps (N × N)
¶
The idea
¶
Take two units (two columns of
rates
). Across all grating trials, do they go up and down
together
? The
Pearson correlation
r answers this:
+1
= perfectly in sync
0
= unrelated
−1
= one goes up when the other goes down
Do this for every pair of N units and you get an
N × N matrix
. It's symmetric (r(A,B) = r(B,A)), with 1s on the diagonal. Plotting it with
plt.imshow
gives the heatmap: each of the N² squares is colored by its r. Same idea as the StackOverflow example (
df.corr()
+
sns.heatmap
).
rates.corr()
is identical to
np.corrcoef(rates.values.T)
.
Three flavors of correlation (this is the key concept)
¶
⚽ Take two strikers on the same team. Their goals per match can be correlated for two very different reasons:
Type
How it's computed
Soccer version
Neuroscience meaning
Total
correlate raw trial-by-trial rates
both score more in some matches than others
a mix of the two below
Signal
correlate their
average
response to each of the 40 conditions (8 directions × 5 TFs)
both score a lot
against weak opponents
: same
preferences
do the two units
like the same stimuli
? (similar tuning)
Noise
subtract each condition's average, then correlate the leftovers
on the
same
opponent, both have an unusually good day
together
: shared form, the crowd, the weather
do they
fluctuate together beyond what the stimulus explains
? This points to
shared input / connectivity
, or shared brain state (arousal, running)
Signal correlations can be large and negative (for opposite preferences). Noise correlations are typically
small (~0.0–0.1) and positive
, so each heatmap gets its own color range.
How to read the image
¶
Units are ordered
V1 first, then the HOA
, and within each area by preferred direction. The black lines split the matrix into blocks:
V1      HOA
     ┌─────────┬─────────┐
 V1  │ V1–V1   │ V1–HOA  │
     ├─────────┼─────────┤
 HOA │ HOA–V1  │ HOA–HOA │
     └─────────┴─────────┘
Diagonal blocks
= correlations
within
an area.
Off-diagonal blocks
=
between
areas (the two are mirror images).
The diagonal itself (always 1) is blanked out so it doesn't dominate the color scale.
🧱 Like a Minecraft map view: each pixel is a pair of units, and its color says how linked they are.

In [ ]:
# grating trials only; units ordered V1 -> HOA, each by preferred direction
order = (tuning.assign(area_rank=tuning.area.map({a: i for i, a in enumerate(AREA_ORDER)}))
                .sort_values(["area_rank", "pref_dir"]).index)
X = rates.loc[~is_blank, order]
conds = [stim.loc[X.index, "direction"], stim.loc[X.index, "tf"]]

total_corr = X.corr()                                             # N x N
signal_corr = X.groupby(conds).mean().corr()                      # correlate 40-condition tuning
noise_corr = (X - X.groupby(conds).transform("mean")).corr()      # correlate trial-to-trial leftovers

n_v1 = (tuning.loc[order, "area"] == "V1").sum()
print(f"{len(order)} units -> {total_corr.shape} correlation matrices ({n_v1} V1, {len(order) - n_v1} {HOA_NAME})")


In [ ]:
def corr_heatmap(C, ax, title):
    C = C.values.copy()
    np.fill_diagonal(C, np.nan)
    vlim = np.ceil(np.nanpercentile(np.abs(C), 99) * 10) / 10     # symmetric color range around 0
    im = ax.imshow(C, cmap="RdBu_r", vmin=-vlim, vmax=vlim, interpolation="nearest")
    for pos in (n_v1 - 0.5,):
        ax.axhline(pos, color="k", lw=1.2)
        ax.axvline(pos, color="k", lw=1.2)
    centers = [n_v1 / 2, n_v1 + (len(C) - n_v1) / 2]
    ax.set_xticks(centers, AREA_ORDER)
    ax.set_yticks(centers, AREA_ORDER)
    ax.set_title(title)
    plt.colorbar(im, ax=ax, shrink=0.75, label="correlation (r)")

fig, axes = plt.subplots(1, 3, figsize=(18, 5.6), constrained_layout=True)
for ax, (C, title) in zip(axes, [(total_corr, "Total correlation"),
                                 (signal_corr, "Signal correlation (tuning similarity)"),
                                 (noise_corr, "Noise correlation (shared fluctuations)")]):
    corr_heatmap(C, ax, title)
fig.suptitle(f"Unit-by-unit correlations during drifting gratings, session {SID} "
             f"(units sorted by area, then preferred direction)")
fig.savefig(FIG_DIR / f"session_{SID}_correlation_heatmaps.png", dpi=200)


Block summary: within-area vs between-area
¶
Average each block, excluding the diagonal, to turn each heatmap into a few numbers you can quote on a slide.

In [ ]:
area_of = tuning.loc[order, "area"].values
blocks = {f"V1–V1": ("V1", "V1"), f"{HOA_NAME}–{HOA_NAME}": (HOA_NAME, HOA_NAME), f"V1–{HOA_NAME}": ("V1", HOA_NAME)}

def block_means(C):
    C = C.values.copy()
    np.fill_diagonal(C, np.nan)
    return {name: np.nanmean(C[np.ix_(area_of == a, area_of == b)]) for name, (a, b) in blocks.items()}

block_tbl = pd.DataFrame({"total": block_means(total_corr),
                          "signal": block_means(signal_corr),
                          "noise": block_means(noise_corr)})
display(block_tbl.round(3))

fig, ax = plt.subplots(figsize=(7, 3.6), constrained_layout=True)
block_colors = [COLORS["V1"], COLORS[HOA_NAME], "#1baf7a"]
w = 0.26
for i, (name, color) in enumerate(zip(block_tbl.index, block_colors)):
    ax.bar(np.arange(3) + (i - 1) * w, block_tbl.loc[name], width=w - 0.02, color=color, label=name)
ax.set_xticks(range(3), ["total", "signal", "noise"])
ax.axhline(0, color=MUTED, lw=0.8)
ax.set(ylabel="mean pairwise r", title="Within-area vs between-area correlation")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.savefig(FIG_DIR / f"session_{SID}_correlation_blocks.png", dpi=200)


15. Interpretation notes for the slides
¶
These are the results from a test run on session 798911424 (Vip-Cre mouse; 94 V1 units, 78 LM units). Re-check them against your own outputs after running.
Tuning (Sections 12–13)
¶
Both areas are strongly driven by gratings.
About 90% of V1 units and 87% of LM units are significantly direction-tuned (ANOVA p < 0.01), with similar median peak rates (~3.9 Hz V1, ~4.3 Hz LM).
Typical selectivity is about the same.
The median OSI is 0.18 in V1 vs 0.20 in LM (Mann-Whitney p = 0.80), and the median DSI is 0.05 vs 0.08 (p = 0.42). How broad a typical unit's tuning is (mean off-peak / peak response) is also about equal: 0.51 vs 0.53.
The difference is in the tail.
V1 has more
highly
orientation-selective units:
19% of V1 units have OSI > 0.5, vs 8% in LM.
You can see this in the ECDF, where the blue curve stays lower (shifted right) above OSI ≈ 0.3.
Most selective units look orientation-tuned, not direction-tuned.
The best example cells have two peaks 180° apart (high OSI, low DSI), and DSI is low in both areas.
Preferred directions tile all 8 directions in both areas.
The diagonal staircase in the population heatmap shows this, so neither area is missing a part of "direction space".
Interpretation:
LM is the first step above V1, and it looks largely like V1 here: it inherits V1-like orientation tuning. The extreme, sharply tuned cells are more concentrated in V1. This fits the idea that LM is "close" to V1 in the hierarchy. Expect a bigger difference if you repeat this with AM or PM.
Correlations (Section 14)
¶
mean pairwise r
V1–V1
LM–LM
V1–LM
total
0.082
0.053
0.054
signal
0.064
0.033
0.032
noise
0.100
0.063
0.070
V1 units are the most correlated with each other, for all three correlation types.
LM–LM pairs are no more correlated than V1–LM pairs.
Signal-correlation heatmap:
sorting by preferred direction produces a
checkerboard
, i.e. bright squares where units prefer the same direction. The same squares continue into the
V1–LM off-diagonal block
. V1 units and LM units that like the same direction respond alike, which is consistent with LM receiving tuned feedforward input from V1.
Noise correlations are small and positive
(~0.06–0.10), the typical range for cortex. The striped rows and columns in the noise heatmap are individual units that co-fluctuate with
many
others. That pattern usually reflects shared brain state (running, arousal) rather than specific wiring.
Caveats to put on the slides
¶
One mouse / one session
(a Vip-Cre line). Run the second session (
762120172
) the same way before claiming any difference between areas.
Temporal frequencies were pooled
to build tuning curves. This blurs units that only respond at one TF and can lower their OSI.
Rates were averaged over 2 s windows.
Slow state changes like running inflate noise correlations;
session.running_speed
could be used to control for this.
QC thresholds
(Allen defaults) and spike-sorting errors affect which units are included.

Part 2b: Same analysis, second session (762120172)
¶
The assignment asks for
2 sessions
. This part re-runs the exact Part 2 analysis on the second session (
762120172
, also V1 + LM).
The only change is
SID
in the first cell.
Explanations are in Part 2 above (CIs, OSI/DSI, signal vs noise correlation), so the headers here are short.
Figures are saved to
dg_spikes/figs/
with this session's ID in the file name.
⚠️ Running these cells
overwrites
Part 2's variables (
rates
,
tuning
,
total_corr
, …) with session 2's. To go back to session 1, re-run Part 2. Section 17 compares the two sessions from the saved files, so it doesn't depend on which one ran last.
Needs
dg_spikes/session_762120172_*
files from section 9 run with
SID = 762120172
.

16a. Load the second session
¶

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

OUT = Path("dg_spikes")
SID = 762120172                          # second specimen (Part 2 used 798911424)
FIG_DIR = OUT / "figs"
FIG_DIR.mkdir(exist_ok=True)

rates = pd.read_pickle(OUT / f"session_{SID}_rates.pkl")               # trials x units (Hz)
units_df = pd.read_csv(OUT / f"session_{SID}_units.csv", index_col=0)  # V1 first, then HOA
stim = pd.read_csv(OUT / f"session_{SID}_stimulus.csv", index_col=0).loc[rates.index]

stim["direction"] = pd.to_numeric(stim.orientation, errors="coerce")  # NaN = blank (gray screen) trial
stim["tf"] = pd.to_numeric(stim.temporal_frequency, errors="coerce")
is_blank = stim.direction.isna()

HOA_NAME = [a for a in units_df.area.unique() if a != "V1"][0]
AREA_ORDER = ["V1", HOA_NAME]
COLORS = {"V1": "#2a78d6", HOA_NAME: "#eb6834"}
MUTED = "#52514e"

assert (rates.columns == units_df.index).all()
print(f"session {SID}: rates table {rates.shape} (trials x units)")
print(units_df.area.value_counts().to_dict(), f"| {(~is_blank).sum()} grating trials, {is_blank.sum()} blank trials")


16b. Tuning curves with 95% CIs: per-unit stats and selectivity
¶

In [ ]:
grat = rates[~is_blank]
dirs_per_trial = stim.loc[grat.index, "direction"]
DIRS = np.sort(dirs_per_trial.unique())                # 0, 45, ..., 315

g = grat.groupby(dirs_per_trial)
tc_mean = g.mean()                                     # directions x units
tc_n = g.size()
t_crit = stats.t.ppf(0.975, tc_n - 1)                  # ~1.99 for n = 75
tc_ci = g.sem().mul(t_crit, axis=0)                    # half-width of the 95% CI
baseline = rates[is_blank].mean()

# selectivity indices from the mean tuning curve
theta = np.deg2rad(DIRS)[:, None]
R = tc_mean.values
osi = np.abs((R * np.exp(2j * theta)).sum(0)) / R.sum(0)
dsi = np.abs((R * np.exp(1j * theta)).sum(0)) / R.sum(0)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    p_anova = [stats.f_oneway(*[grat.loc[dirs_per_trial == d, u] for d in DIRS]).pvalue for u in rates.columns]

tuning = pd.DataFrame({"area": units_df.area,
                       "pref_dir": DIRS[R.argmax(0)],
                       "peak_rate": R.max(0),
                       "baseline": baseline,
                       "osi": osi, "dsi": dsi,
                       "p_anova": p_anova}, index=rates.columns)
tuning["tuned"] = tuning.p_anova < 0.01
tuning.round(3).head()


Example units
¶

In [ ]:
def plot_tuning(unit, ax, color, title=True):
    # mean rate at each direction with 95% CI error bars, blank-trial baseline dashed
    ax.errorbar(DIRS, tc_mean[unit], yerr=tc_ci[unit], color=color, lw=2,
                marker="o", ms=5, capsize=3, elinewidth=1.2)
    ax.axhline(baseline[unit], color=MUTED, ls="--", lw=1)
    ax.set_xticks(DIRS[::2])
    ax.set_xlim(-20, 335)
    ax.set_ylim(bottom=0)
    ax.spines[["top", "right"]].set_visible(False)
    if title:
        ax.set_title(f"unit {unit}\nOSI {tuning.osi[unit]:.2f} · DSI {tuning.dsi[unit]:.2f}", fontsize=9)

N_EX = 4
fig, axes = plt.subplots(2, N_EX, figsize=(3.2 * N_EX, 6), constrained_layout=True)
for row, area in zip(axes, AREA_ORDER):
    pool = tuning[(tuning.area == area) & tuning.tuned & (tuning.peak_rate >= 2)]
    for ax, unit in zip(row, pool.osi.nlargest(N_EX).index):
        plot_tuning(unit, ax, COLORS[area])
    row[0].set_ylabel(f"{area}\nfiring rate (Hz)")
for ax in axes[-1]:
    ax.set_xlabel("grating direction (deg)")
fig.suptitle(f"Example tuning curves, session {SID} (mean ± 95% CI; dashed = blank-screen baseline)")
fig.savefig(FIG_DIR / f"session_{SID}_tuning_examples.png", dpi=200)


Every unit (saved to
dg_spikes/figs/
)
¶

In [ ]:
def tuning_grid(area, ncols=10):
    ids = tuning[tuning.area == area].sort_values("pref_dir").index
    nrows = int(np.ceil(len(ids) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(1.9 * ncols, 1.6 * nrows),
                             sharex=True, constrained_layout=True)
    for ax in axes.flat[len(ids):]:
        ax.axis("off")
    for ax, unit in zip(axes.flat, ids):
        plot_tuning(unit, ax, COLORS[area], title=False)
        ax.set_title(f"{unit}{'' if tuning.tuned[unit] else ' (n.s.)'}", fontsize=7)
        ax.tick_params(labelsize=6)
    fig.suptitle(f"{area}: all {len(ids)} units, session {SID} (mean ± 95% CI vs direction; "
                 f"dashed = baseline; n.s. = not significantly tuned)")
    fig.savefig(FIG_DIR / f"session_{SID}_tuning_all_{area}.png", dpi=150)
    return fig

for area in AREA_ORDER:
    tuning_grid(area)


16c. Population tuning heatmap
¶

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5), constrained_layout=True)
for ax, area, cmap in zip(axes, AREA_ORDER, ["Blues", "Oranges"]):
    ids = tuning[tuning.area == area].sort_values("pref_dir").index
    norm_tc = (tc_mean[ids] / tc_mean[ids].max()).T              # units x directions, peak = 1
    im = ax.imshow(norm_tc, aspect="auto", cmap=cmap, vmin=0, vmax=1, interpolation="nearest")
    ax.set_xticks(range(len(DIRS)), [f"{d:.0f}" for d in DIRS])
    ax.set(title=f"{area} ({len(ids)} units)", xlabel="grating direction (deg)",
           ylabel="units (sorted by preferred direction)")
    fig.colorbar(im, ax=ax, label="rate / unit's peak rate", shrink=0.8)
fig.suptitle(f"Population tuning, session {SID}")
fig.savefig(FIG_DIR / f"session_{SID}_tuning_population.png", dpi=200)


Selectivity: V1 vs LM
¶

In [ ]:
summary = tuning.groupby("area").agg(n_units=("osi", "size"),
                                     pct_tuned=("tuned", "mean"),
                                     median_peak_hz=("peak_rate", "median"),
                                     median_osi=("osi", "median"),
                                     median_dsi=("dsi", "median")).loc[AREA_ORDER]
summary["pct_tuned"] = (100 * summary.pct_tuned).round(0)
display(summary.round(3))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), constrained_layout=True)
for ax, metric, label in zip(axes, ["osi", "dsi"], ["orientation selectivity (OSI)", "direction selectivity (DSI)"]):
    a, b = (tuning.loc[tuning.area == ar, metric].dropna() for ar in AREA_ORDER)
    p = stats.mannwhitneyu(a, b).pvalue
    for area, vals in zip(AREA_ORDER, (a, b)):
        x = np.sort(vals)
        ax.step(x, np.arange(1, len(x) + 1) / len(x), where="post", color=COLORS[area], lw=2,
                label=f"{area} (median {np.median(x):.2f})")
    ax.set(xlabel=label, ylabel="fraction of units ≤ x", title=f"Mann-Whitney p = {p:.3g}", ylim=(0, 1.02))
    ax.legend(frameon=False, loc="lower right")
    ax.spines[["top", "right"]].set_visible(False)
fig.savefig(FIG_DIR / f"session_{SID}_selectivity.png", dpi=200)


16d. Correlation matrices (total, signal, noise)
¶

In [ ]:
# grating trials only; units ordered V1 -> HOA, each by preferred direction
order = (tuning.assign(area_rank=tuning.area.map({a: i for i, a in enumerate(AREA_ORDER)}))
                .sort_values(["area_rank", "pref_dir"]).index)
X = rates.loc[~is_blank, order]
conds = [stim.loc[X.index, "direction"], stim.loc[X.index, "tf"]]

total_corr = X.corr()                                             # N x N
signal_corr = X.groupby(conds).mean().corr()                      # correlate 40-condition tuning
noise_corr = (X - X.groupby(conds).transform("mean")).corr()      # correlate trial-to-trial leftovers

n_v1 = (tuning.loc[order, "area"] == "V1").sum()
print(f"{len(order)} units -> {total_corr.shape} correlation matrices ({n_v1} V1, {len(order) - n_v1} {HOA_NAME})")


Correlation heatmaps
¶

In [ ]:
def corr_heatmap(C, ax, title):
    C = C.values.copy()
    np.fill_diagonal(C, np.nan)
    vlim = np.ceil(np.nanpercentile(np.abs(C), 99) * 10) / 10     # symmetric color range around 0
    im = ax.imshow(C, cmap="RdBu_r", vmin=-vlim, vmax=vlim, interpolation="nearest")
    for pos in (n_v1 - 0.5,):
        ax.axhline(pos, color="k", lw=1.2)
        ax.axvline(pos, color="k", lw=1.2)
    centers = [n_v1 / 2, n_v1 + (len(C) - n_v1) / 2]
    ax.set_xticks(centers, AREA_ORDER)
    ax.set_yticks(centers, AREA_ORDER)
    ax.set_title(title)
    plt.colorbar(im, ax=ax, shrink=0.75, label="correlation (r)")

fig, axes = plt.subplots(1, 3, figsize=(18, 5.6), constrained_layout=True)
for ax, (C, title) in zip(axes, [(total_corr, "Total correlation"),
                                 (signal_corr, "Signal correlation (tuning similarity)"),
                                 (noise_corr, "Noise correlation (shared fluctuations)")]):
    corr_heatmap(C, ax, title)
fig.suptitle(f"Unit-by-unit correlations during drifting gratings, session {SID} "
             f"(units sorted by area, then preferred direction)")
fig.savefig(FIG_DIR / f"session_{SID}_correlation_heatmaps.png", dpi=200)


Block summary: within-area vs between-area
¶

In [ ]:
area_of = tuning.loc[order, "area"].values
blocks = {f"V1–V1": ("V1", "V1"), f"{HOA_NAME}–{HOA_NAME}": (HOA_NAME, HOA_NAME), f"V1–{HOA_NAME}": ("V1", HOA_NAME)}

def block_means(C):
    C = C.values.copy()
    np.fill_diagonal(C, np.nan)
    return {name: np.nanmean(C[np.ix_(area_of == a, area_of == b)]) for name, (a, b) in blocks.items()}

block_tbl = pd.DataFrame({"total": block_means(total_corr),
                          "signal": block_means(signal_corr),
                          "noise": block_means(noise_corr)})
display(block_tbl.round(3))

fig, ax = plt.subplots(figsize=(7, 3.6), constrained_layout=True)
block_colors = [COLORS["V1"], COLORS[HOA_NAME], "#1baf7a"]
w = 0.26
for i, (name, color) in enumerate(zip(block_tbl.index, block_colors)):
    ax.bar(np.arange(3) + (i - 1) * w, block_tbl.loc[name], width=w - 0.02, color=color, label=name)
ax.set_xticks(range(3), ["total", "signal", "noise"])
ax.axhline(0, color=MUTED, lw=0.8)
ax.set(ylabel="mean pairwise r", title="Within-area vs between-area correlation")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.savefig(FIG_DIR / f"session_{SID}_correlation_blocks.png", dpi=200)


17. Comparing the two sessions
¶
One mouse can be a fluke; a pattern in
both
mice is much more convincing. ⚽ It's like judging a striker on two different seasons instead of one: if they're top scorer both times, it's probably skill, not luck.
The cell below recomputes the key numbers for
both
sessions straight from
dg_spikes/
, the same way Parts 2 and 2b do:
Selectivity:
median OSI, and the % of highly selective units (OSI > 0.5), per area
Correlations:
mean within-V1, within-LM and V1–LM correlation, for each of total / signal / noise
For the slides, look for
what stays the same across sessions
(a real V1 vs LM difference) vs
what flips
(individual variability, or too few units).

In [ ]:
SESSIONS = [798911424, 762120172]

def session_summary(sid):
    # recompute selectivity and block correlations for one session from the saved files
    r = pd.read_pickle(OUT / f"session_{sid}_rates.pkl")
    u = pd.read_csv(OUT / f"session_{sid}_units.csv", index_col=0)
    st = pd.read_csv(OUT / f"session_{sid}_stimulus.csv", index_col=0).loc[r.index]
    direction = pd.to_numeric(st.orientation, errors="coerce")
    tf = pd.to_numeric(st.temporal_frequency, errors="coerce")
    blank = direction.isna()
    grat, d = r[~blank], direction[~blank]

    tc = grat.groupby(d).mean()
    th = np.deg2rad(tc.index.values)[:, None]
    osi = pd.Series(np.abs((tc.values * np.exp(2j * th)).sum(0)) / tc.values.sum(0), index=r.columns)
    sel = (pd.DataFrame({"area": u.area, "osi": osi})
             .groupby("area").osi.agg(n_units="size", median_osi="median",
                                      pct_osi_over_0_5=lambda x: 100 * (x > 0.5).mean()))

    conds = [d, tf[~blank]]
    mats = {"total": grat.corr(),
            "signal": grat.groupby(conds).mean().corr(),
            "noise": (grat - grat.groupby(conds).transform("mean")).corr()}
    area = u.loc[r.columns, "area"].values
    rows = []
    for kind, C in mats.items():
        C = C.values.copy()
        np.fill_diagonal(C, np.nan)
        for name, (a, b) in blocks.items():
            rows.append({"session": sid, "kind": kind, "block": name,
                         "r": np.nanmean(C[np.ix_(area == a, area == b)])})
    return sel.assign(session=sid), pd.DataFrame(rows)

sel_all, corr_all = zip(*[session_summary(s) for s in SESSIONS])
sel_all = pd.concat(sel_all).reset_index().set_index(["session", "area"])
corr_all = pd.concat(corr_all)

print("Selectivity per session and area")
display(sel_all.round(3))
print("Mean pairwise correlation per block")
display(corr_all.pivot_table(index=["kind", "session"], columns="block", values="r")
                .loc[["total", "signal", "noise"], list(blocks)].round(3))


In [ ]:
# one panel per correlation type; color = block (same as the bar charts), marker = session
markers = dict(zip(SESSIONS, ["o", "s"]))
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True, constrained_layout=True)
for ax, kind in zip(axes, ["total", "signal", "noise"]):
    sub = corr_all[corr_all.kind == kind]
    for i, (name, color) in enumerate(zip(blocks, block_colors)):
        for j, sid in enumerate(SESSIONS):
            r = sub[(sub.block == name) & (sub.session == sid)].r.iloc[0]
            ax.scatter(i + (j - 0.5) * 0.18, r, s=90, color=color, marker=markers[sid],
                       edgecolor="white", linewidth=1.5, zorder=3)
    ax.set_xticks(range(len(blocks)), list(blocks))
    ax.axhline(0, color=MUTED, lw=0.8)
    ax.set_title(f"{kind} correlation")
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("mean pairwise r")
for sid in SESSIONS:
    axes[-1].scatter([], [], marker=markers[sid], color=MUTED, s=70, label=f"session {sid}")
axes[-1].legend(frameon=False, loc="lower right")
fig.suptitle("Within- vs between-area correlation, both sessions")
fig.savefig(FIG_DIR / "both_sessions_correlation_blocks.png", dpi=200)


18. Interpretation notes: both sessions
¶
These are the numbers from a test run on both sessions. Both mice are Vip-Cre; the analysis covers V1 + LM. Re-check them against your outputs after running.
Selectivity
¶
Session 798911424
Session 762120172
units (V1 / LM)
94 / 78
84 / 74
% tuned (V1 / LM)
90% / 87%
87% / 88%
median OSI (V1 / LM)
0.18 / 0.20 (p = 0.80)
0.16 / 0.13 (
p = 0.02
)
% highly selective, OSI > 0.5 (V1 / LM)
19% / 8%
8% / 0%
median DSI (V1 / LM)
0.05 / 0.08
0.07 / 0.08
✅
Consistent in both mice:
almost all units in both areas are tuned, and
the most sharply orientation-selective units live in V1
. In session 2, LM has
no
units with OSI > 0.5, and V1's median OSI is also significantly higher.
✅
Consistent:
direction selectivity is low and similar in both areas. Most tuned cells care about stripe
angle
(two peaks 180° apart), not motion direction.
⚠️ The difference in
typical
(median) selectivity only reaches significance in one session. So the safe claim is "V1 has more highly selective cells", not "every V1 cell is sharper".
Interpretation: LM inherits V1-like orientation tuning but loses the sharpest selectivity. This fits LM being one step up the hierarchy, where cells pool over several V1 inputs.
Side note: some LM units fire
below
their blank-screen baseline for most directions; the gratings suppress them. Session 2 has an example (unit 951878253).
Correlations
¶
mean pairwise r
V1–V1
LM–LM
V1–LM
total, s1 / s2
0.082 / 0.064
0.053 /
0.093
0.054 / 0.063
signal, s1 / s2
0.064 / 0.054
0.033 /
0.061
0.032 / 0.048
noise, s1 / s2
0.100
/ 0.077
0.063 /
0.098
0.070 / 0.072
❌
Not consistent: which area is more correlated flips between mice.
V1–V1 is highest in session 1, but LM–LM is highest in session 2. With two mice we
can't
claim that one area is more internally correlated than the other.
✅
Consistent:
the between-area block (V1–LM) is never the most correlated. The top block is always a within-area one.
✅
Consistent:
noise correlations are small and positive (~0.06–0.10) in every block, the typical range for cortex.
✅
Consistent:
the signal-correlation heatmaps show the same checkerboard in both sessions. Units preferring the same direction respond alike,
including across V1 and LM
, which fits tuned input flowing from V1 to LM.
Possible reasons the within-area pattern flips: how much the mouse ran or how aroused it was in each session (shared state boosts noise correlations), how many units and what depths each probe sampled, and how well the V1 and LM receptive fields overlapped (section 6 plot).
Suggested slide takeaways
¶
Similar:
both areas are strongly driven and orientation-tuned; preferred directions cover all 8 directions; noise correlations are small and positive.
Different:
V1 has more highly selective neurons in both mice, and LM's tuning is somewhat broader.
Unclear:
whether V1 or LM is more internally correlated; this depends on the mouse. A next step would be more sessions, or controlling for running speed (
session.running_speed
).
Caveats:
only 2 mice, both Vip-Cre; temporal frequencies were pooled; rates were averaged over 2 s windows; Allen QC with presence ratio > 0.95.